# Semantic interpretation: approved pilot and candidate counts

This notebook preserves the four owner-confirmed development decisions in a separate worksheet, then performs the cue screen specified in `docs/METHODS.md`. The decisions were confirmed after discussion, not independent blind review. The cue rules retrieve possible material; they do not assign semantic labels or establish corpus prevalence. No new text examples are displayed and no models are fitted.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
from IPython.display import display

current = Path.cwd()
ROOT = next((p for p in [current, *current.parents]
             if (p / "data/processed/model_corpus_v1.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or notebooks directory.")
source = ROOT / "data/processed/model_corpus_v1.csv"
source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
reference = json.loads((ROOT / "results/followup_audit_v1/manifest.json").read_text())
if source_hash != reference["source_sha256"]:
    raise ValueError("Prepared corpus hash differs from the frozen input.")
corpus = pd.read_csv(source, keep_default_na=False)
prepared = corpus.loc[corpus["preparation_status"].eq("ready_for_exploration")].copy()
if len(prepared) != 479 or not prepared["execution_number"].is_unique:
    raise ValueError("Unexpected prepared screening population.")
print("Screening population: 479 prepared documents; includes two TF-IDF zero-feature documents.")
print("One wholly indirect summary remains excluded by the existing preparation status.")


## Confirmed development decisions

The source of an embedded claim and the action subject inside it are different fields. These four decisions describe expressed propositions, not private states or actual guilt. The earlier blank/editable worksheet is retained. A different reviewed file will be created only after its source fields have been checked.


In [ ]:
pilot_path = ROOT / "data/processed/semantic_pilot_development_v1.csv"
if not pilot_path.exists():
    raise FileNotFoundError("Run notebook 05 first to create the pilot worksheet.")
pilot = pd.read_csv(pilot_path, keep_default_na=False)
decisions_path = ROOT / "reports/semantic_pilot_decisions_v1.csv"
decisions = pd.read_csv(decisions_path, keep_default_na=False)
if len(pilot) != 4 or not pilot["claim_id"].is_unique or not decisions["claim_id"].is_unique:
    raise ValueError("Unexpected pilot size or duplicate claim IDs.")
if set(pilot["claim_id"]) != set(decisions["claim_id"]):
    raise ValueError("Pilot IDs differ from approved decisions.")
decisions = decisions.set_index("claim_id").loc[pilot["claim_id"]].reset_index()
for column in ["claim_id", "execution_number", "claim_text", "context_sha256", "corpus_sha256"]:
    if not pilot[column].astype(str).equals(decisions[column].astype(str)):
        raise ValueError(f"Approved decisions differ from worksheet source field: {column}")
for _, row in pilot.iterrows():
    context = corpus.loc[corpus["execution_number"].eq(row["execution_number"]), "model_text"].iloc[0]
    if row["context_text"] != context or context[int(row["span_start"]):int(row["span_end"])] != row["claim_text"]:
        raise ValueError("Worksheet context or offsets do not match frozen corpus.")
    if row["corpus_sha256"] != source_hash or hashlib.sha256(context.encode()).hexdigest() != row["context_sha256"]:
        raise ValueError("Pilot source hash mismatch.")
annotation_columns = ["claim_source", "event_type", "event_actor", "polarity", "review_note"]
reviewed = pilot.drop(columns=annotation_columns).merge(
    decisions[["claim_id"] + annotation_columns + ["review_process", "decision_recorded_date", "use_status"]],
    on="claim_id", validate="one_to_one", how="left",
)
display(reviewed[["claim_id", "claim_source", "event_type", "event_actor", "polarity"]])


## Literal cue screening

All 479 prepared documents are screened, including texts without retained TF-IDF features. Cues may overlap. Action words can refer to execution, religion or another person; a saying cue does not establish that someone else is being quoted. Candidate means an action/innocence cue or a selected negated-knowledge cue matched. No keyword result is a human label. Nonhits can still contain relevant paraphrases or indirect references.


In [ ]:
sys.path.insert(0, str(ROOT / "src"))
from semantic_screening import PATTERNS, screen_documents
screened = screen_documents(prepared)
flag_columns = list(PATTERNS) + ["candidate_document"]
summary = pd.DataFrame({
    "documents": [int(screened[column].sum()) for column in flag_columns],
}, index=flag_columns)
display(summary)
print("Noncandidate documents:", int((~screened["candidate_document"]).sum()))
possible_combinations = [
    f"action={a};knowledge={k};reporting={r}"
    for a in [0, 1] for k in [0, 1] for r in [0, 1]
]
combinations = (
    screened.groupby("cue_combination").size()
    .reindex(possible_combinations, fill_value=0)
    .rename("documents").to_frame()
)
assert int(combinations["documents"].sum()) == 479
display(combinations)
print("Counts describe rule hits, not authentic semantic classes or independent evaluation data.")


## Save separate screening artifacts

New ignored files are created under `data/processed/`. Existing files are accepted only if bytes are identical; differing files are not overwritten. The protocol and exact regexes, module hash and input hash are recorded. These artifacts do not modify the original pilot, frozen annotations, corpus or model outputs.


In [ ]:
def save_new_or_identical(path, content):
    if path.exists():
        if path.read_bytes() != content:
            raise FileExistsError(f"Existing artifact differs; preserve it and use a new version: {path.name}")
        print("Verified existing:", path.relative_to(ROOT))
    else:
        with path.open("xb") as handle:
            handle.write(content)
        print("Created:", path.relative_to(ROOT))

output = ROOT / "data/processed"
save_new_or_identical(output / "semantic_pilot_reviewed_v1.csv",
                     reviewed.to_csv(index=False).encode("utf-8-sig"))
save_new_or_identical(output / "semantic_candidate_screen_v1.csv",
                     screened.to_csv(index=False).encode("utf-8-sig"))
metadata = {
    "source_sha256": source_hash,
    "screening_module_sha256": hashlib.sha256((ROOT / "src/semantic_screening.py").read_bytes()).hexdigest(),
    "protocol_sha256": hashlib.sha256((ROOT / "docs/SEMANTIC_SCREENING_PROTOCOL.md").read_bytes()).hexdigest(),
    "approved_decisions_sha256": hashlib.sha256(decisions_path.read_bytes()).hexdigest(),
    "prepared_documents": len(prepared),
    "candidate_documents": int(screened["candidate_document"].sum()),
    "noncandidate_documents": int((~screened["candidate_document"]).sum()),
    "patterns": PATTERNS,
    "matching": "Case-insensitive literal cue regexes on unmodified prepared text",
    "purpose": "Feasibility retrieval only; no semantic labels or model performance",
    "review_process": "Four pilot decisions confirmed after AI-assisted discussion; not blind",
}
save_new_or_identical(output / "semantic_screen_manifest_v1.json",
                     (json.dumps(metadata, indent=2) + "\n").encode("utf-8"))
print("Original pilot, frozen annotations, corpus and model results were not modified.")
